---
title: "B-lineage coverage mapping to immune subniches"
author: "Codex analysis workflow"
date: today
format:
  html:
    toc: true
    code-fold: true
    embed-resources: true
---

This report maps the frozen 2,669-Bin50 B-lineage RNA coverage mask onto the frozen K=3/4/5 immune-subniche assignments. It does not rerun RCTD, construct neighborhoods, cluster bins, select K, recalibrate markers, or compare diseases.

Exact render command: `tmux new-session -d -s b_subniche_report 'bash scripts/1056_render_b_lineage_subniche_mapping_report.sh'`.

In [ ]:
from pathlib import Path
import json
import os
import platform
import sys

import matplotlib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

project_root = Path(os.environ['B_SUBNICHE_PROJECT_ROOT'])
result_dir = Path(os.environ['B_SUBNICHE_RESULT_DIR'])
summary = pd.read_csv(result_dir / '03_SUBNICHE_B_ENRICHMENT_SUMMARY.tsv', sep='\t')
patient = pd.read_csv(result_dir / '02_PATIENT_SUBNICHE_B_COVERAGE.tsv', sep='\t')
transitions = pd.read_csv(result_dir / '04_K345_LINEAGE_TRANSITIONS.tsv', sep='\t')
audit = pd.read_csv(result_dir / '08_FINAL_AUDIT.tsv', sep='\t')
assert audit['passed'].all()
assert len(patient) == 244
display(Markdown(f'**Frozen result directory:** `{result_dir}`'))
display(audit)

## Main result

Each chip contains a B-enriched branch that persists across K resolutions. K=3 is the coarse reference; K=4/5 show how the same branch is retained or split at higher resolution. Cluster labels are chip- and K-specific and are not cross-chip identities.

In [ ]:
recurrent = summary[summary['mapping_class'].str.startswith('recurrent_B')].copy()
columns = [
    'chip', 'k', 'cluster', 'candidate_bin_n', 'b_coverage_bin_n',
    'b_coverage_fraction', 'fraction_of_chip_b_coverage',
    'patient_direction_enriched_n', 'median_patient_enrichment_ratio',
    'core_corroborated_fraction_of_b', 'extended_gain_fraction_of_b',
    'largest_patient_contribution', 'mapping_class',
]
display(recurrent[columns].style.format({
    'b_coverage_fraction': '{:.1%}',
    'fraction_of_chip_b_coverage': '{:.1%}',
    'median_patient_enrichment_ratio': '{:.2f}',
    'core_corroborated_fraction_of_b': '{:.1%}',
    'extended_gain_fraction_of_b': '{:.1%}',
    'largest_patient_contribution': '{:.1%}',
}))
display(Image(filename=str(result_dir / 'figures/01_B_ENRICHMENT_BY_SUBNICHE.png'), alt='Heatmap of median patient B-lineage enrichment ratios across chips, K resolutions, and immune subniches'))
display(Image(filename=str(result_dir / 'figures/02_PATIENT_B_COVERAGE_BY_SUBNICHE.png'), alt='Patient-level B-lineage coverage fractions across chips, K resolutions, and immune subniches'))

## Cross-resolution lineage

The dominant B-containing transitions are L3 K3 I1 → K4 I4 → K5 I3, J2 K3 I3 → K4 I1 → K5 I3, and Y40102K8 K3 I1 → K4 I4 → K5 I3. The transition table quantifies overlap without asserting that similarly numbered clusters are equivalent across chips.

In [ ]:
dominant_rows = []
for chip in ['L3', 'J2', 'Y40102K8']:
    for parent_k in [3, 4]:
        local = transitions[(transitions['chip'] == chip) & (transitions['parent_k'] == parent_k)]
        dominant_rows.append(local.sort_values('b_coverage_bin_n', ascending=False).iloc[0])
dominant = pd.DataFrame(dominant_rows)
display(dominant[['chip', 'parent_k', 'parent_cluster', 'child_k', 'child_cluster', 'overlap_bin_n', 'b_coverage_bin_n', 'b_coverage_fraction', 'patient_n']].style.format({'b_coverage_fraction': '{:.1%}'}))

## Interpretation and limitations

- L3 shows the most consistent separation: its B branch captures 74.5% of the chip B mask at K=3 and has a median patient enrichment ratio of 3.08.
- J2 has a recurrent but more distributed B pattern: the K=3 branch captures 45.7% of the chip B mask with a median patient enrichment ratio of 1.91.
- Y40102K8 concentrates nearly all B coverage in one lineage, but most B-positive bins come from one patient; it remains patient-imbalanced.
- All recurrent mappings retain a patient-imbalance warning. Patient-level direction, not Bin-level pseudo-replication, is the evidence unit.
- B-lineage coverage is RNA support, not a claim that every Bin is a pure B cell. These mapping labels are not final immune-subniche names. Final naming should wait for lineage-specific T, NK, Plasma, Myeloid, and Mast/Basophil coverage audits.

In [ ]:
print('Session information')
print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('matplotlib:', matplotlib.__version__)
print('Result directory:', result_dir)
